Provide semantic documents for specific path differences and determine which regulatory dimensions are triggered by those path differences.

Traverse the first-level directories under `dataset4geodiff\experiment_results\experiments\2024-Nov\results\final\callpath-rq12-patterns\USA-Others-androzoo-all`. Each directory is named after an apkname; open each directory and load its TXT document containing semantic descriptions of path differences between APK versions.

Load the clause JSONL document `dataset4clauses\assessment_code_prompts\clause_prompts_lines.jsonl`.

Then perform coarse classification across five major buckets: `[P, CR, R, E, O]`. First determine which buckets are highly likely and which are definitely unlikely for each semantic path-difference document. Then perform fine-grained classification within the selected buckets.

Return JSON with exactly these top-level fields:
{
  "likely": ["P"|"CR"|"C"|"R"|"E"|"O", ...],
  "unlikely": [...],
  "unsure": [...],
  "reason": "..."
}

Fine-grained classification: iterate through the clause prompts in each selected bucket. For example, if the previous step identifies the document as belonging to bucket P, select prompts beginning with P, then iterate through all TXT files, query the OpenAI API in batches, and save the results in batches.
Return JSON with exactly these top-level fields:
{
  "apkname": string,
  {
    "semantic_diff": "xx",
    "candidate_clauses": [
        {
          "Clause_id": string,
          "confidence": "high|medium|low",
          "score": number,
          "why": string
        },
        {
          "Clause_id": string,
          "confidence": "high|medium|low",
          "score": number,
          "why": string
        },
        ...
      ],
  }
}
If no triggered clause is found, return `candidate_clauses.Clause_id` as null.

Configuration: directories, clause-prompt loading, and grouping by bucket

The `clause_prompts_lines` file needs a prompt covering `content`. Add one prompt specifying whether content must be provided.

Then create a dedicated clause-prompt JSON object for this task, revise the task description, and remove the return-format section if it is unnecessary.

In [1]:
# {f_position}\{s_position}
f_position = "AA1_first_batch"
s_position = "AA4_forth_100_batch" # AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch  ## AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch

# {parameter}
# parameter first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1
parameter = "first_4"

There are many folders under the ROOT directory, each named after an APK. Keep only the folders whose names appear in `apkname_list.csv`.

In [2]:
import os, json, re, time, random
from pathlib import Path
import pandas as pd

ROOT = Path(r"dataset4geodiff\experiment_results\experiments\2024-Nov\results\final\callpath-rq12-patterns\USA-Others-androzoo-all")
PROMPTS_JSONL = Path(r"dataset4clauses\geodiff_clasues_prompt\clause_prompts_lines_geodiff.jsonl")

apkname_csv_fp = Path(
    rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10"
    rf"\{f_position}\{s_position}\merged_two_matrix\apkname_list.csv"
)

# first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1

OUT_DIR = ROOT.parent / "out_openai_geodiff_clause_mapping/first_4" #5_26
OUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_BUCKETS = ["P", "CR", "C", "R", "E", "O"]

def load_clause_prompts(jsonl_path: Path) -> dict[str, str]:
    out = {}
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if len(obj) != 1:
                raise ValueError(f"Bad jsonl line (expect single key): {line[:80]}")
            clause_id, prompt = next(iter(obj.items()))
            out[str(clause_id).strip()] = str(prompt)
    return out

def clause_bucket(clause_id: str) -> str:
    m = re.match(r"^([A-Za-z]+)\d+", clause_id.strip())
    if not m:
        return "O"
    return m.group(1).upper()

clause_id_to_prompt = load_clause_prompts(PROMPTS_JSONL)
bucket_to_clauses = {b: [] for b in VALID_BUCKETS}
for cid in sorted(clause_id_to_prompt.keys()):
    b = clause_bucket(cid)
    if b not in bucket_to_clauses:
        b = "O"
    bucket_to_clauses[b].append(cid)

{b: len(v) for b, v in bucket_to_clauses.items()}

d:\softwall_install\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


{'P': 20, 'CR': 5, 'C': 1, 'R': 11, 'E': 5, 'O': 0}

Parse each APK TXT file into multiple semantic_diff entries.

In [ ]:
def extract_semantic_diffs_(txt: str) -> list[str]:
    txt = txt.strip()
    if not txt:
        return []

    # 1) JSON format (this is the format of Claude_pattern_summary.txt).
    try:
        obj = json.loads(txt)
        diffs = []

        def walk(x):
            if isinstance(x, dict):
                if "original_entry" in x and isinstance(x["original_entry"], str):
                    cat = x.get("category")
                    if isinstance(cat, str) and cat.strip():
                        diffs.append(f"[{cat.strip()}] {x['original_entry'].strip()}")
                    else:
                        diffs.append(x["original_entry"].strip())
                for v in x.values():
                    walk(v)
            elif isinstance(x, list):
                for it in x:
                    walk(it)

        walk(obj)
        diffs = [d for d in diffs if d]
        if diffs:
            return diffs
    except Exception:
        pass

    # 2) Plain-text fallback.
    return [txt]

def extract_semantic_diffs(txt_path: str) -> list[str]:
    with open(txt_path, 'r', encoding='utf-8') as file:
        content = file.read()
        print(content)
    return content

OpenAI: coarse classification (5 buckets) + fine-grained classification (run only the selected buckets)

In [4]:
COARSE_SYSTEM = """
You are a classifier. You will be given a semantic description of code/path changes across app versions.

Your job is to classify which regulatory clause buckets are likely relevant.

Use the following bucket definitions, which reflect the actual clause groups in the taxonomy:

- P = Personal-data processing details.
  This bucket includes statements about:
  * categories or specific items of personal data collected, used, shared, disclosed, or sold;
  * purposes of collection, use, sharing, disclosure, or sale;
  * whether data provision is mandatory or optional;
  * methods of collection, processing, storage, retention, deletion, destruction, or de-identification;
  * retention/storage period;
  * sources of sensitivity such as sensitive data, biometric data, children's data;
  * cross-border transfer details;
  * whether providing data is required by law, contract, or necessity to enter a contract.

- CR = Controller / representative / contact / source-recipient information.
  This bucket includes statements about:
  * identity of the controller;
  * controller representative or local representative;
  * contact details of the controller or DPO;
  * address/place of establishment of the representative;
  * categories or sources from which personal data are collected;
  * categories of recipients to whom personal data are disclosed, transferred, or shared.

- C = Consent disclosure / Consent presentation.
    This bucket includes statements about:
    * whether consent is obtained;
    * categories of personal data for which consent is obtained;
    * purposes for which consent is obtained;
    * method of obtaining consent;

- R = User or consumer rights.
  This bucket includes statements about:
  * right to access / know / confirm processing;
  * right to know purposes of processing;
  * right to know third-party recipients or transfer recipients;
  * right to consent or confirm;
  * right to delete / erase;
  * right to correct / rectify;
  * right to object;
  * right to restrict or limit processing/use/disclosure;
  * right to withdraw consent;
  * right to data portability;
  * right to lodge a complaint;
  * right to know what data is sold/shared/disclosed and to whom;
  * right to opt out;
  * right to contest solely automated decision-making / profiling;
  * right to seek compensation or damages;
  * general statements of privacy rights, if they clearly grant one of the above rights.

- E = Rights-exercise procedures and compliance mechanisms.
  This bucket includes statements about:
  * how to appeal a controller's decision;
  * how the business verifies a request to know, delete, or correct;
  * how an authorized agent may submit requests;
  * parental/guardian verification for child-related consent;
  * opt-in process for minors;
  * how opt-out preference signals are processed;
  * how consumers can implement opt-out preference signals in a frictionless manner;
  * procedural or operational mechanisms for carrying out privacy rights or compliance obligations.


Classification rules:
1. Assign a bucket if the semantic description is plausibly relevant to at least one clause in that bucket.
2. This is a coarse routing step, so prefer recall over precision.
3. A semantic description may belong to multiple buckets.
4. Do NOT force "O" when another bucket clearly applies.
5. Distinguish carefully:
   - R = the existence of a user/consumer right;
   - E = the procedure or mechanism for exercising or implementing that right;
   - CR = who the controller is, how to contact them, where data comes from, or who receives it;
   - C = whether and how consent is obtained;
   - P = what data is handled, why, how, how long, under what conditions, and related processing details.

Output requirements:
Return JSON only, in the following format:
{
  "likely": ["P","CR"], 
  "unlikely": ["R","E", "C"],
  "reason": "1-2 sentences"
}

Valid bucket labels are only:
["P", "CR", "C", "R", "E"]

"""

In [ ]:
        # Return JSON with exactly:
        # {
        # "presence_flag": 0 or 1,
        # "confidence": "high"|"medium"|"low",
        # "score": number,
        # "why": string
        # }

In [ ]:
# Semantic code-diff description!!!

# Then save it as JSON!!!

In [5]:
FINE_SYSTEM = """
        You are a knowledgeable, helpful, and honest assistant. 
        You have deep expertise in current privacy regulations, 
        including the California Consumer Privacy Act (CCPA) for California, 
        the Texas Data Privacy and Security Act (TDPSA) for Texas, 
        the General Data Protection Regulation (GDPR) for Europe, 
        the Digital Personal Data Protection Act, 2023 (DPDP) for India, 
        the Nigeria Data Protection Act, 2023 (NDPA) for Nigeria, 
        the Personal Data Protection Law No. 151 of 2020 (PDPL) for Egypt, 
        the Personal Data Protection Law (PDPL) for Saudi Arabia, 
        Decree 13/2023/ND-CP on Personal Data Protection (PDPD) for Vietnam, 
        and the Law on the Protection of Personal Data No. 6698 (KVKK/LPPD) for Turkey. 
        You have strong skills in analyzing and explaining semantic description of code/path changes across app versions, ensuring clarity and accuracy in interpreting compliance requirements.
        You will be given:
        (1) a clause-checking prompt (written for privacy policy text)
        (2) a semantic code-diff description
        Your job is to judge whether the semantic diff plausibly indicates the clause is triggered.
        Output requirements:
        Return JSON only, in the following format:
        {
            "semantic_diff": "xx",
            "candidate_clauses": [
                {
                "Clause_id": string,
                "presence_flag": 0 or 1,
                "why": string
                }
                {
                "Clause_id": string,
                "presence_flag": 0 or 1,
                "why": string
                },
                ...
            ],
        }

        Rules:
        - For presence_flag, set to '1' if the semantic code-diff description includes relevant information, or '0' if it does not.
        - if the presence_flag is 0, return none.
        """

In [ ]:
# <!-- api_key="" -->

# <!-- ttu
# #  -->

In [ ]:
from openai import OpenAI
client = OpenAI(api_key="") 



def openai_json(messages, model="gpt-5.1", temperature=0):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        response_format={"type": "json_object"},
    )
    raw = (resp.choices[0].message.content or "").strip()
    
    # Try to parse JSON; preserve the original text if parsing fails.
    try:
        return {"ok": True, "output": json.loads(raw), "raw": raw}
    except Exception:
        return {"ok": False, "output": None, "raw": raw, "error": "output_not_json"}

def coarse_classify(semantic_diff: str) -> dict:

    # # Call only the relevant clause prompts to reduce cost.
    messages = [
            {"role": "system", "content": COARSE_SYSTEM},
            {"role": "user", "content": semantic_diff},
        ]
    coarse = openai_json(messages, temperature=0)
    print(f"Success" if coarse["ok"] else f"Failure: {coarse.get('error')}")
    print(f"    Raw output:{coarse['output']}")
    coarse_api_data = coarse['output'] or {}

    # Simulated valid JSON string returned by the API.
    # coarse = '{"likely": ["P","C"], "unlikely": ["R","E"], "unsure": ["O"], "reason": "Simulated coarse classification result"}'
    # coarse_api_data = json.loads(coarse)

    return coarse_api_data

def eval_clause_on_diff(clause_id: str, clause_prompt: str, semantic_diff: str) -> dict:
    # # Call only the relevant clause prompts to reduce cost.
    user = (
        f"Clause_id: {clause_id}\n\n"
        f"Clause prompt:\n{clause_prompt}\n\n"
        f"Semantic diff:\n{semantic_diff}\n"
    )
    messages = [
            {"role": "system", "content": FINE_SYSTEM},
            {"role": "user", "content": user},
        ]
    out = openai_json(messages, temperature=0)
    print(f"Success" if out["ok"] else f"Failure: {out.get('error')}")
    print(f"    Raw output:{out['output']}")
    api_data = out['output'] or {}

    # Simulated valid JSON string returned by the API.
    # coarse = '{"likely": ["P","CR"], "unlikely": ["R","E"], "unsure": ["O"], "reason": "Simulated coarse classification result"}'
    # out = '{"semantic_diff": "xx","candidate_clauses": [{"Clause_id": "string","presence_flag": 1,"confidence": "high","score": 0.9,"why": "string"}, {"Clause_id": "string","presence_flag": 1,"confidence": "medium","score": 0.6,"why": "string"}]}'
    # api_data = json.loads(out)

    return api_data

Main loop: iterate through APK folders -> read TXT files -> coarse classification -> fine-grained clause classification -> write result JSON

In [ ]:
apk_dirs = [p for p in ROOT.iterdir() if p.is_dir()]
print("apk dirs:", len(apk_dirs))

# Read apkname_list.csv.
apkname_df = pd.read_csv(apkname_csv_fp)

# Assume the column is named apkname.
target_apknames = set(apkname_df["apkname"].dropna().astype(str))

print("target apkname count:", len(target_apknames))

# Select only folders under ROOT whose names appear in the CSV.
apk_selected_dirs = [
    p for p in ROOT.iterdir()
    if p.is_dir() and p.name in target_apknames
]

print("matched apk dirs:", len(apk_selected_dirs))

apk dirs: 1115
target apkname count: 15
matched apk dirs: 15


In [ ]:
def build_candidate_clause_obj(clause_id: str | None, presence_flag: str, why: str) -> dict:
    return {
        "Clause_id": clause_id,  # None is allowed.
        "presence_flag": presence_flag,
        "why": why,
    }

apk_dirs = [p for p in ROOT.iterdir() if p.is_dir()]
print("apk dirs:", len(apk_dirs))

# Load apkname_list.csv.
apkname_df = pd.read_csv(apkname_csv_fp)
target_apknames = set(apkname_df["apkname"].dropna().astype(str))

# Select only directories under ROOT whose names appear in the CSV.
apk_selected_dirs = [
    p for p in ROOT.iterdir()
    if p.is_dir() and p.name in target_apknames
]
print("apk selected dirs:", len(apk_selected_dirs))
count = 1

for apk_dir in sorted(apk_selected_dirs):

    # if count ==2:
    #     break

    apkname = apk_dir.name
    out_path = OUT_DIR / f"{apkname}.json"
    if out_path.exists():
        continue

    txt_files = sorted(apk_dir.glob("*.txt"))
    if not txt_files:
        continue

    # The workflow expects one TXT file; use the first. Extend here to merge multiple TXT files if needed.
    txt = txt_files[0].read_text(encoding="utf-8", errors="ignore")
    obj = json.loads(txt)
    # print("obj\n", obj)

    results = []
    
    coarse = coarse_classify(txt)
    likely = [b for b in coarse.get("likely", []) if b in VALID_BUCKETS]
    print(f"Coarse classification for {apkname}: likely buckets: {likely}, reason: {coarse.get('reason', '')}")

    results.append({
        "coarse": coarse
    })

    if not likely:
        likely = ["O"]

    triggered = []
    for b in likely:
        # if count == 2:
        #         break
        for clause_id in bucket_to_clauses.get(b, []):
            # if count == 2:
            #     break
            clause_prompt = clause_id_to_prompt[clause_id]
            r = eval_clause_on_diff(clause_id, clause_prompt, txt)
            print(r)
            # Iterate through the results.
            for clause in r.get("candidate_clauses", []):
                # if count == 4:
                #     break
                clause_id = clause.get("Clause_id")
                presence_flag = clause.get("presence_flag")
                # score = clause.get("score")
                # confidence = clause.get("confidence")
                why = clause.get("why")
                print(f"presence_flag: {presence_flag}")
                print(f"Found Clause_id: {clause_id}")

                if int(presence_flag) == 1:
                    triggered.append(
                        build_candidate_clause_obj(
                            clause_id=clause_id,
                            presence_flag=presence_flag,
                            why=why,
                        )
                    )
                # count += 1
                time.sleep(random.uniform(0.2, 0.6))  # Brief rate limiting to prevent 429 responses.
            count += 1

    if not triggered:
        triggered = [build_candidate_clause_obj(None, "0.0", "No clause triggered.")]

    results.append({
        "path_diff_semantic": obj,
        "candidate_clauses": triggered
    })

    out_obj = {
        "apkname": apkname,
        "results": results
    }
    print("out_obj\n",json.dumps(out_obj, ensure_ascii=False, indent=2))
    out_path.write_text(json.dumps(out_obj, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[OK]", apkname, "->", out_path)

<!-- Apply a relaxed threshold to candidate_clauses.score in output.json to obtain a set of possible mappings. -->

In [ ]:
# No need to perform rescore / rerank.

# Directly apply a relaxed threshold to candidate_clauses.score in prompt2_output.json to obtain a set of possible mappings:

# Recommended thresholds:
# >= 0.55: possible mapping
# 0.40 ~ 0.55: weak mapping
# For this example:
# PC002 -> P11, P10; P4 and CR6 are weaker candidates
# PC003 -> P12
# PC001 / PC004 -> none

# This is sufficient for a baseline version.

simple process the rs, copy it to new directory

In [12]:
import json
from pathlib import Path
import os
from typing import Dict, Any, List, Set, Optional

In [13]:
def postprocess_clause_mapping(
    prompt_json_path,
    output_json_path=None,
    # strong_threshold=0.55,
    # weak_threshold=0.40
):
    """
    后处理 prompt2 输出，得到 path -> clause 的宽松映射关系

    规则：
    - score >= strong_threshold        -> possible_mapping
    - weak_threshold <= score < strong_threshold -> weak_mapping
    - score < weak_threshold          -> discard
    """

    # prompt_json_path = Path(prompt_json_path)

    with open(prompt_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = {
        "apkname": data.get("apkname"),
        "bucket": data.get("results", [{}])[0].get("coarse", {}),
        # "thresholds": {
        #     # "possible_mapping_threshold": strong_threshold,
        #     # "weak_mapping_threshold": weak_threshold
        # },
        "path_mapping_results": []
    }

    for path_item in data.get("results", []):
        # path_change_id = path_item.get("path_change_id")
        # bucket_predictions = path_item.get("coarse", [])
        candidate_clauses = path_item.get("candidate_clauses", [])

        # possible_mappings = []
        # weak_mappings = []

        mapping = []

        for clause in candidate_clauses:
            score = clause.get("score", 0)

            kept_clause = {
                "ours_id": clause.get("Clause_id"),
                "presence_flag": clause.get("presence_flag"),

                "why": clause.get("why")
            }


            mapping.append(kept_clause)

        # all_kept_mappings = possible_mappings + weak_mappings
        all_kept_mappings = mapping

        result_item = {
            "all_kept_mappings": all_kept_mappings,
        }

        results["path_mapping_results"].append(result_item)
    print(results)
    if output_json_path is not None:
        output_json_path = Path(output_json_path)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    return results
# if __name__ == "__main__":
#     input_path = r"dataset4geodiff\experiment_results\experiments\2024-Nov\results\final\callpath-rq12-patterns\out_openai_geodiff_clause_mapping\com.brainium.solitairefree.json"
#     output_path = r"dataset4geodiff\experiment_results\experiments\2024-Nov\results\final\callpath-rq12-patterns\out_openai_geodiff_clause_mapping\com.brainium.solitairefree_process.json"

#     results = postprocess_clause_mapping(
#         prompt_json_path=input_path,
#         output_json_path=output_path,
#     )

#     print(json.dumps(results, ensure_ascii=False, indent=2))

In [14]:
# parameter first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1


if __name__ == "__main__": #dataset4geodiff\experiment_results\experiments\2024-Nov\results\final\callpath-rq12-patterns\out_openai_geodiff_clause_mapping\5_26
    input_directory = Path(rf"dataset4geodiff\\experiment_results\\experiments\\2024-Nov\\results\\final\\callpath-rq12-patterns\\out_openai_geodiff_clause_mapping\{parameter}")
    clauses_file_path = Path(r"dataset4clauses\\geodiff_clasues_prompt\\clause_prompts_lines_geodiff.jsonl") # PROMPTS_JSONL = Path(r"dataset4clauses\\geodiff_clasues_prompt\\clause_prompts_lines_geodiff.jsonl")
    # output_directory = input_directory

    output_directory = Path(rf"dataset4geodiff\\out_openai_sematic_geodiff_txt\{parameter}")
    folders  = os.listdir(input_directory)
    # 仅遍历第一层文件夹
    for folder in folders:
        # print(f"Found folder: {folder}")
        input_full_path  = Path(f"{input_directory}/{folder}")
        # print(f"Processing folder: {input_full_path}")


        if not folder.endswith(".json"):
            continue
        output_dir= output_directory / folder.replace(".json", "")
        output_dir.mkdir(parents=True, exist_ok=True)

        input_path = input_full_path
        output_path = os.path.join(output_dir, folder.replace(".json", "_process.json")) # clauses.json
        print(f"Input path: {input_path}")
        print(f"Output path: {output_path}")

        results = postprocess_clause_mapping(
            prompt_json_path=input_path,
            output_json_path=output_path,
        )

        print(json.dumps(results, ensure_ascii=False, indent=2))

Input path: dataset4geodiff\experiment_results\experiments\2024-Nov\results\final\callpath-rq12-patterns\out_openai_geodiff_clause_mapping\first_4\air.com.spilgames.TrollFaceQuestVideoGames2.json
Output path: dataset4geodiff\out_openai_sematic_geodiff_txt\first_4\air.com.spilgames.TrollFaceQuestVideoGames2\air.com.spilgames.TrollFaceQuestVideoGames2_process.json
{'apkname': 'air.com.spilgames.TrollFaceQuestVideoGames2', 'bucket': {'likely': [], 'unlikely': ['P', 'CR', 'C', 'R', 'E'], 'reason': 'The changes relate to authentication flow and session/UI management timing, without mentioning personal data, consent, user rights, or controller/contact details.'}, 'path_mapping_results': [{'all_kept_mappings': []}, {'all_kept_mappings': [{'ours_id': None, 'presence_flag': '0.0', 'why': 'No clause triggered.'}]}]}
{
  "apkname": "air.com.spilgames.TrollFaceQuestVideoGames2",
  "bucket": {
    "likely": [],
    "unlikely": [
      "P",
      "CR",
      "C",
      "R",
      "E"
    ],
    "rea

In [ ]:
# def batch_process_directory(
#     input_full_path: str,
#     apk_name: str,
#     clauses_path: str,
#     output_dir: str
# ) -> None:
#     # output_dir = input_full_path

#     for filename in os.listdir(input_full_path):
#         if not filename.endswith("_processed_converted.json"):
#             continue

#         in_path = os.path.join(input_full_path, filename)
#         out_path = os.path.join(output_dir, filename.replace("_processed_converted.json", "_final_clauses_mapping.json")) # clauses.json

        

#         try:
#             process_one_app_diff_file(
#                 input_full_dir= input_full_path,
#                 input_json_path=in_path,
#                 clauses_path=clauses_path,
#                 output_json_path=out_path,
#                 apkname=apk_name
#             )
#             print(f"[OK] {filename}")
#         except Exception as e:
#             print(f"[ERROR] {filename}: {e}")

# if __name__ == "__main__":
#     # input_directory = "dataset4geodiff\out_openai_sematic_geodiff_txt"
#     clauses_file_path = "dataset4geodiff\clauses.json"
#     # output_directory = input_directory

#     input_directory = Path(r"dataset4geodiff\out_openai_sematic_geodiff_txt")

#     # 仅遍历第一层文件夹
#     for folder in sorted([p for p in input_directory.iterdir() if p.is_dir()]):
#         input_full_path  = input_directory / folder.name
#         # print(f"Processing folder: {input_full_path}")
#         batch_process_directory(
#             input_full_path=input_full_path,
#             apk_name=folder.name,
#             clauses_path=clauses_file_path,
#             output_dir=input_full_path
#         )